# Anonimize fMRI (HCL) 

In [ ]:
import matplotlib
#enable detached figure
matplotlib.use('TkAgg')

from module.utils import *
import glob

#set path
path = "/home/caredda/Videos/DATA_TO_UPLOAD/"
#patient ID
array_Patient_id = np.array(["P35", "P51", "P59", "P60", "P61", "P64", "P75", "P77", "P78", "P83"])

for Patient_id in array_Patient_id:
    # Get fMRI filenames
    filename_fMRI = glob.glob(path+Patient_id+"/stats/*.nii")


    #text position in MRI slices
    l1 = 118
    l2 = 592
    c1 = 11
    c2 = 389

    #Create output directory
    output_dir = path+Patient_id+"/stats/anonym/"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    #Loop over fMRI files
    for MRI_filename in filename_fMRI:
        print(MRI_filename)

        #Load fMRI data
        nib_img = nib.load(MRI_filename)
        vol = np.asanyarray(nib_img.dataobj)
        rgb_array = np.stack([vol['R'], vol['G'], vol['B']], axis=-1)
        


        plt.close('all')
        # data = rgb_array[:,:,-1,:]
        # plt.figure()
        # plt.imshow(data, cmap='gray')
        # plt.axis('off')
        # plt.title("fMRI data")
        # plt.show(block=True)


        #Init output volumes
        fMRI_mask = np.zeros(rgb_array.shape[:3], dtype=np.uint8)
        anatomical_vol = np.zeros(rgb_array.shape[:3], dtype=np.uint8)

        #Loop over slices
        for i in range(rgb_array.shape[2]):

            data = rgb_array[:,:,i,:]

            # Mask text in slices
            data[:,0:c1] = 0
            data[:,c2:] = 0
            data[0:l1,:] = 0
            data[l2:,:] = 0

            #Get fMRI mask (red-to-yellow pixels)
            fMRI_mask[:,:,i] = (data[:,:,0] != data[:,:,1]) | (data[:,:,1] != data[:,:,2]) | (data[:,:,2] != data[:,:,0])
            anatomical_vol[:,:,i] = data[:,:,0]


            # plt.figure()
            # plt.subplot(131)
            # plt.imshow(data, cmap='gray')
            # plt.axis('off')
            # plt.title("fMRI data")
            # plt.subplot(132)
            # plt.imshow(fMRI_mask[:,:,i], cmap='gray')
            # plt.axis('off')
            # plt.title("fMRI mask")
            # plt.subplot(133)
            # plt.imshow(anatomical_vol[:,:,i], cmap='gray')
            # plt.axis('off')
            # plt.title("Anatomical mask")
            # plt.show()

        fMRI_mask = fMRI_mask.astype(np.uint8)

        # Create new nii volumes with original affine and header
        nib_anatomical_vol = nib.Nifti1Image(anatomical_vol, nib_img.affine, nib_img.header)
        nib_fMRI_mask = nib.Nifti1Image(fMRI_mask, nib_img.affine, nib_img.header)

        nib_anatomical_vol.set_data_dtype(np.uint8)
        nib_fMRI_mask.set_data_dtype(np.uint8)


        # Save to new file
        nib.save(nib_anatomical_vol, output_dir + os.path.basename(MRI_filename).split('.nii')[0] + '_anatomical.nii')
        nib.save(nib_fMRI_mask, output_dir + os.path.basename(MRI_filename).split('.nii')[0] + '_fMRI_stats.nii')



# Functions

In [ ]:
import matplotlib
#enable detached figure
matplotlib.use('TkAgg')

from module.utils import *
import glob
import cv2
from skimage.metrics import normalized_mutual_information, normalized_root_mse

def normalized_cross_correlation(img1, img2):
    img1 = img1.astype(np.float32)
    img2 = img2.astype(np.float32)

    mean1 = img1.mean()
    mean2 = img2.mean()
    std1 = img1.std()
    std2 = img2.std()

    ncc = np.mean((img1 - mean1) * (img2 - mean2)) / (std1 * std2 + 1e-8)
    return ncc

def create_itk_transverse_slice(vol_itk, id_slice, h_flip = False):
    np_vol = itk.GetArrayFromImage(vol_itk)

    #Create transverse slice
    np_transverse_slice = np_vol[id_slice,:,:]
    if h_flip:
        np_transverse_slice = np.flip(np_transverse_slice, axis=1)
    transverse_slice  = itk.GetImageFromArray(np.ascontiguousarray(np_transverse_slice))
    transverse_slice .SetSpacing([vol_itk.GetSpacing()[1], vol_itk.GetSpacing()[2]])

    return transverse_slice

def create_np_transverse_slice(vol_itk, id_slice, h_flip = False):
    np_vol = itk.GetArrayFromImage(vol_itk)

    #Create transverse slice
    np_transverse_slice = np_vol[id_slice,:,:]
    if h_flip:
        np_transverse_slice = np.flip(np_transverse_slice, axis=1)
    

    return np_transverse_slice



def affine_registration(imgRef, imgReg):

    #normalize images
    img_ref = (255*(imgRef - imgRef.min()) / (imgRef.max() - imgRef.min())).astype(np.uint8)
    img_to_reg = (255*(imgReg - imgReg.min()) / (imgReg.max() - imgReg.min())).astype(np.uint8)




    # 1. Detect and compute keypoints and descriptors
    orb = cv2.ORB_create(5000)
    kp_ref, des_ref = orb.detectAndCompute(img_ref, None)
    kpB, desB = orb.detectAndCompute(img_to_reg, None)

    # 2. Match descriptors using BFMatcher (brute force)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des_ref, desB)

    # Sort matches by distance (best first)
    matches = sorted(matches, key=lambda x: x.distance)

    # 3. Extract point correspondences
    ptsA = np.float32([kp_ref[m.queryIdx].pt for m in matches[:50]])
    ptsB = np.float32([kpB[m.trainIdx].pt for m in matches[:50]])

    # 4. Estimate affine transform
    M, inliers = cv2.estimateAffine2D(ptsB, ptsA)  # or estimateAffinePartial2D

    # 5. Warp imageB onto imageA
    registered_B = cv2.warpAffine(imgReg, M, (img_ref.shape[1], img_ref.shape[0]))

    return registered_B









def rigid_register_2d(fixed, moving, metric_func, thresh_val = 0, log = False, other_moving = None):

    #normalize images
    norm_fixed = 255*(fixed - fixed.min()) / (fixed.max() - fixed.min())
    norm_moving = 255*(moving - moving.min()) / (moving.max() - moving.min())


    # Get mask images
    mask_fixed = norm_fixed > thresh_val
    mask_moving = norm_moving > thresh_val

    mask_fixed = mask_fixed.astype(np.uint8)*255
    mask_moving = mask_moving.astype(np.uint8)*255

    if mask_fixed.sum() == 0 or mask_moving.sum() == 0:
        # print("Warning: One of the images has no valid pixels, skipping registration.")
        return moving, 0, None

    # Find contours
    contours_fixed, _ = cv2.findContours(mask_fixed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_moving, _ = cv2.findContours(mask_moving, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Get the bounding box of the largest contour
    largest_contour_fixed = max(contours_fixed, key=cv2.contourArea)
    largest_contour_moving = max(contours_moving, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour_fixed)
    x_m, y_m, w_m, h_m = cv2.boundingRect(largest_contour_moving)

    if (w<10) or (h<10):
        # print("Warning: Fixed image is too small, skipping registration.")
        return moving, 0, None
    
    # Crop the images to the bounding box
    moving_cropped = moving[y_m:y_m+h_m, x_m:x_m+w_m]
    fixed_cropped = fixed[y:y+h, x:x+w]

    if log:
        print("moving", w_m, h_m )
        print("fixed", w, h )





    # Resize the moving image to the size of the fixed image
    moving_resized = cv2.resize(moving_cropped, (w, h), interpolation=cv2.INTER_NEAREST)


    #normalize images
    norm_fixed = 255*(fixed_cropped - fixed_cropped.min()) / (fixed_cropped.max() - fixed_cropped.min())
    norm_moving = 255*(moving_resized - moving_resized.min()) / (moving_resized.max() - moving_resized.min())

    size_k = 13
    norm_fixed = cv2.GaussianBlur(norm_fixed.astype(np.uint8), (size_k, size_k), 0)

    # norm_moving = cv2.GaussianBlur(norm_fixed.astype(np.uint8), (5, 5), 0)



    # plt.figure()
    # plt.suptitle("Cropped images")
    # plt.subplot(121)
    # plt.imshow(norm_fixed, cmap='gray')
    # plt.title("Fixed cropped")
    # plt.axis('off')
    # plt.subplot(122)
    # plt.imshow(norm_moving, cmap='gray')
    # plt.title("Moving cropped")
    # plt.axis('off')
    # plt.show()

    # Calculate the metric
    metric = metric_func(norm_fixed, norm_moving)


    #Apply transformation to other moving image if provided
    out_img = None
    if other_moving is not None:
        other_moving_resized = cv2.resize(other_moving[y_m:y_m+h_m, x_m:x_m+w_m], (w, h), interpolation=cv2.INTER_NEAREST)
        out_img = np.zeros_like(fixed)
        out_img[y:y+h, x:x+w] = other_moving_resized

    return norm_moving, metric, out_img



def get_fMRI_slice_ID_in_FLAIR(transverse_slice_fMRI, transverse_slice_fMRI_stats, vol_itk_FLAIR, start_slice_id=0, thresh_val=10):


    metric_func = normalized_cross_correlation
    metric = np.array([])

    #Loop over slices
    for slice_id_FLAIR in range(start_slice_id,vol_itk_FLAIR.shape[0]):
        #Create transverse slice
        transverse_slice_FLAIR  = create_np_transverse_slice(vol_itk_FLAIR, slice_id_FLAIR)

        #Process registration
        img_registered, metric_val, _ =  rigid_register_2d(transverse_slice_FLAIR, transverse_slice_fMRI, metric_func,thresh_val)
        metric = np.append(metric, metric_val) 

    #Apply registration to the correct slice
    transverse_slice_FLAIR  = create_np_transverse_slice(vol_itk_FLAIR, metric.argmax())
    img_registered, metric_val, out_fMRI_stats =  rigid_register_2d(transverse_slice_FLAIR,
                                                    transverse_slice_fMRI,
                                                    metric_func,
                                                    thresh_val,
                                                    other_moving=transverse_slice_fMRI_stats)

    return start_slice_id+metric.argmax(), out_fMRI_stats


# Find the slice in MRI that matches anatomical MRI

In [ ]:
#set path
path = "/home/caredda/Videos/DATA_TO_UPLOAD/"




plt.close('all')
thresh_val = 20

#patient ID
# array_Patient_id = np.array(["P35", "P51", "P59", "P60", "P61", "P64", "P75", "P77", "P78", "P83"])
Patient_id = "P35"

data_path = path+Patient_id+"/stats/anonym/"

# Get fMRI filenames
filename_fMRI = glob.glob(data_path+"*_anatomical.nii")
MRI_filename = filename_fMRI[0]

print("MRI filename: ", MRI_filename)

# Load anatomical MRI volumes
vol_itk_fMRI = itk.imread(MRI_filename, itk.F)
vol_itk_fMRI_stats = itk.imread(MRI_filename.split("_anatomical.nii")[0]+"_fMRI_stats.nii", itk.F)
vol_itk_FLAIR = itk.imread(path+Patient_id+"/stats/FLAIR.nii", itk.F)



#Create output fMRI overlay having FLAIR shape
np_fMRI_overlay = np.zeros(vol_itk_FLAIR.shape)
# print("fMRI overlay shape: ", np_fMRI_overlay.shape)


#fMRI slice idx in FLAIR
# slice_idx_in_FLAIR = []





#Create transverse slice
transverse_slice_fMRI  = create_np_transverse_slice(vol_itk_fMRI, 0, h_flip = True)
transverse_slice_fMRI_stats  = create_np_transverse_slice(vol_itk_fMRI_stats, 0, h_flip = True)

#Get slice ID in FLAIR
id_slice_FLAIR, img_registered = get_fMRI_slice_ID_in_FLAIR(transverse_slice_fMRI, transverse_slice_fMRI_stats,
                                                             vol_itk_FLAIR, start_slice_id = 0, thresh_val=thresh_val)


# transverse_slice_FLAIR = create_np_transverse_slice(vol_itk_FLAIR, id_slice_FLAIR)
# print("First ID ",id_slice_FLAIR)
# slice_idx_in_FLAIR.append(id_slice_FLAIR)

#reconstruct fMRI overlay
np_fMRI_overlay[id_slice_FLAIR,:,:] = img_registered

# plt.figure()
# plt.subplot(221)
# plt.imshow(transverse_slice_FLAIR, cmap='gray')
# plt.axis('off')
# plt.title("Transverse slice from FLAIR")

# plt.subplot(222)
# plt.imshow(transverse_slice_fMRI, cmap='gray')
# plt.axis('off')
# plt.title("Transverse slice from anatomical used in fMRI")

# plt.subplot(223)
# plt.imshow(img_registered, cmap='gray')
# plt.axis('off')
# plt.title("Registered fMRI")

# # plt.subplot(224)
# # plt.plot(metric)
# # plt.title("Registration metric")
# # plt.xlabel("Slice ID")
# # plt.ylabel("Metric value")
# plt.show(block=True)





for id_slice_fMRI in range(1,vol_itk_fMRI.shape[0]):
    #Create transverse slice
    transverse_slice_fMRI  = create_np_transverse_slice(vol_itk_fMRI, id_slice_fMRI, h_flip = True)
    transverse_slice_fMRI_stats  = create_np_transverse_slice(vol_itk_fMRI_stats, id_slice_fMRI, h_flip = True)

    #Get slice ID in FLAIR
    id_slice_FLAIR, img_registered = get_fMRI_slice_ID_in_FLAIR(transverse_slice_fMRI, transverse_slice_fMRI_stats, 
                                                                vol_itk_FLAIR, start_slice_id = id_slice_FLAIR+1, thresh_val=thresh_val)
    # slice_idx_in_FLAIR.append(id_slice_FLAIR)

    #reconstruct fMRI overlay
    np_fMRI_overlay[id_slice_FLAIR,:,:] = img_registered


# print(slice_idx_in_FLAIR)

#Write fMRI overlay to file
fMRI_overlay = itk.GetImageFromArray(np.ascontiguousarray(np_fMRI_overlay))
fMRI_overlay.SetSpacing(vol_itk_FLAIR.GetSpacing())
fMRI_overlay.SetDirection(vol_itk_FLAIR.GetDirection())
fMRI_overlay.SetOrigin(vol_itk_FLAIR.GetOrigin())

# Write result
itk.imwrite(fMRI_overlay, data_path + "fMRI_overlay.nii")
